# SPECIEUROPE v3.0 cleanup

Cleans the [SPECIEUROPE](https://source-apportionment.jrc.ec.europa.eu/Specieurope/index.aspx) v3.0 database so that it contains individual chemicals with standardized identifiers. SPECIEUROPE is a database of European air pollution source profiles, designed for source apportionment and emission analysis, with species encoding aligned to the US EPA SPECIATE system. It contains ca. 200 particulate chemical profiles (organic and inorganic) derived from measurements, source apportionment studies, composites, and theoretical estimates.

**Workflow**

| Step | What happens |
| --- | --- |
| 1 | Load the SPECIEUROPE CSV and rename columns to follow other similar databases (e.g., `Relative Mass` to `weight_fraction`) |
| 2 | Clean names and CAS values (odd characters, synonyms) and remove mixtures |
| 3 | Format CAS numbers (`cas`) and ensure text format to avoid unwanted Excel conversion |
| 4 | Retrieve compound properties (InChIKey, title, IUPAC name) from PubChem; resumable, cached |
| 5 | Optionally save the cleaned intermediate table |
| 6 | Merge the PubChem data into the table |
| 7 | Remove entries without InChIKey, by InChIKey list and by keyword; sort and save |
| 8-9 | Descriptive statistics and export per compound |

**Settings**

All user-editable settings are in the configuration cell:

- `CSV_PATH`: the raw SPECIEUROPE CSV (`Source 999 - all sources.csv`, see `original_databases/README.md` for how it was downloaded).
- `OUTPUT_DIR` / `CACHE_DIR`: output (`specieurope_cleaned`) and cache (`cache`) folders, created next to the notebook.
- `LOOKUP_LIMIT`: set to e.g. 50 for a quick test run of the PubChem lookups; `None` runs all compounds.
- `MAX_REQ_PER_SEC` and `N_WORKERS`: PubChem request rate (PubChem requests <= 5 requests/second) and number of parallel workers.

**Requirements:** Python 3 with `pandas`, `numpy`, `chardet` and `requests`, run in JupyterLab (or another Jupyter front end); internet access is needed for the PubChem step. Run the notebook cell by cell. Step 4 can be interrupted and re-run; finished lookups are kept in the cache file.

**Output files** (written to `specieurope_cleaned/`)

| File | Description |
| --- | --- |
| `SPECIEUROPE_Profiles_All_01_Fltrd.csv` | Cleaned table before InChIKey/keyword filtering (step 5) |
| `SPECIEUROPE_Profiles_All_02_Fltrd.csv` | Final filtered table (step 7b) |
| `SPECIEUROPE_Profiles_All_03_CAS_Profile_Composition.csv` | Per-compound statistics (step 9) |

The PubChem lookup cache is written to `cache/PubChem_lookup_cache.csv`.

**License:** MIT; intended for academic and research use.

In [12]:
# Reset env to erase previous vars, dicts, dfs, etc.
%reset -f

from datetime import datetime

print(f"Environment reset | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

Environment reset | 2026-11-18 22:17:54



In [13]:
# Configuration & imports
import os
import re
import time
import io
from collections import defaultdict
from datetime import datetime

import numpy as np
import pandas as pd

# Shared helpers/filters (common lives in the parent folder of this notebook)
import sys
sys.path.insert(0, os.path.abspath(".."))
from common.utils import print_overwrite, df_stats
from common.compound_filters import apply_standard_filters
from common.pubchem import PubChemClient, lookup_attempts, per_row, run_cached_lookup
from common.cleaning import add_uid, clean_names, count_odd_cas, normalize_cas, read_cache

# ---------------------------------------------------------------------------
# CONFIGURATION (all user-editable settings live here)
# ---------------------------------------------------------------------------
DATA_DIR = "../../../data"
# Input: SPECIEUROPE CSV (downloaded from the JRC SPECIEUROPE website)
CSV_PATH = os.path.join(DATA_DIR, "source_databases/Source 999 - all sources.csv")

# Output & cache folders (subfolders of the notebook folder)
OUTPUT_DIR = os.path.join(DATA_DIR, "cleaned_source_databases/specieurope/cleaned")
CACHE_DIR = os.path.join(DATA_DIR, "cleaned_source_databases/specieurope/cache")
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

# Cache file (resumable PubChem lookups, one row per unique identifier set)
PUBCHEM_CACHE_CSV = os.path.join(CACHE_DIR, "PubChem_lookup_cache.csv")

# Output files
INTERMEDIATE_CSV = os.path.join(OUTPUT_DIR, "SPECIEUROPE_Profiles_All_01_Fltrd.csv")   # cleaned table before InChIKey/keyword filtering (step 5)
FLTRD_CSV = os.path.join(OUTPUT_DIR, "SPECIEUROPE_Profiles_All_02_Fltrd.csv")          # final filtered table (step 7b)
STATS_CSV = os.path.join(OUTPUT_DIR, "SPECIEUROPE_Profiles_All_03_CAS_Profile_Composition.csv")  # per-compound statistics (step 9)

# PubChem enrichment settings
LOOKUP_LIMIT = None      # set to e.g. 50 for a quick test run; None = all compounds
MAX_REQ_PER_SEC = 4      # PubChem asks for <= 5 requests/second. You may be blocked when increasing the rate.
N_WORKERS = 4
CHUNK = 200              # compounds per cache write; smaller chunks give a more responsive ETA

print(f"Configuration & imports done | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

Configuration & imports done | 2026-11-18 22:17:54



In [14]:
# Helper to remove previous progress message
# print_overwrite is imported from inquire_common.utils in the configuration cell

print(f"Helper to remove previous progress message created | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")

Helper to remove previous progress message created | 2026-11-18 22:17:54



## Step 1: Load the SPECIEUROPE CSV and rename columns

Paths and settings are defined in the configuration cell above. Confirm that the CSV exists before reading it.

In [15]:
# Check that the SPECIEUROPE CSV exists (CSV_PATH is set in the configuration cell)
assert os.path.exists(CSV_PATH), f"CSV file not found: {CSV_PATH}"
print("SPECIEUROPE CSV path is valid.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")

SPECIEUROPE CSV path is valid. 2026-11-18 22:17:54 



Read the CSV into `df`.

The downloaded file has two quirks:

- **Mixed encoding:** most non-ASCII characters are UTF-8 (e.g., the ligature `ﬁ` and `μ`), but others are single Windows-1252 bytes (e.g., `ó`, `±`, `ß`). The file is therefore neither valid UTF-8 nor valid Windows-1252, and `chardet` (which only sees the first 100 kB) reports plain ASCII. Each run of non-ASCII bytes is decoded as UTF-8 where possible and as Windows-1252 otherwise.
- **Trailing comma:** every line ends with a comma, which creates an empty last column (dropped in the next cell). `index_col=False` stops pandas from using the first column as an index.

In [16]:
# Read CSV file (mixed UTF-8 / Windows-1252 encoding)
print("Reading CSV with pandas.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'))

def _decode_run(match):
    """Decode a run of non-ASCII bytes: UTF-8 sequences where valid, Windows-1252 for any other byte."""
    b, out, i = match.group(), [], 0
    while i < len(b):
        for n in (4, 3, 2):
            try:
                out.append(b[i:i + n].decode("utf-8"))
                i += n
                break
            except UnicodeDecodeError:
                pass
        else:
            out.append(b[i:i + 1].decode("cp1252", errors="replace"))
            i += 1
    return "".join(out).encode("utf-8")

with open(CSV_PATH, "rb") as f:
    raw = f.read()
text = re.sub(rb"[\x80-\xff]+", _decode_run, raw).decode("utf-8")
print(f"Undecodable characters: {text.count(chr(0xFFFD)):,}")

df = pd.read_csv(io.StringIO(text), index_col=False, low_memory=False)
del raw, text

print("CSV loaded.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'))
print(f"\nCSV shape: \n{df.shape[0]:,} rows × {df.shape[1]} columns\n")

Reading CSV with pandas. 2026-11-18 22:17:54
Undecodable characters: 0
CSV loaded. 2026-11-18 22:17:54

CSV shape: 
13,148 rows × 22 columns



Rename columns to follow other similar databases (e.g., `Relative Mass` to `weight_fraction`, `Specie` to `orig_name`, `Id` to `PROFILE_CODE`). Empty columns (the one created by the trailing commas) are dropped. Columns missing from the file are skipped silently, so check the printed list.

In [17]:
# Drop empty columns and rename columns
print(f"Input SPECIEUROPE shape: \n{df.shape[0]:,} rows × {df.shape[1]} columns\n")

print(f"Input SPECIEUROPE columns:")
for col in df.columns:
    print(col)
print()

# Empty columns (e.g. 'Unnamed: 21' from the trailing comma on every line)
empty_cols = [c for c in df.columns if df[c].isna().all()]
df = df.drop(columns=empty_cols)
print(f"Dropped empty columns: {empty_cols}\n")

# Rename columns
df = df.rename(columns={
    k: v for k, v in {
        "Id": "PROFILE_CODE",
        "Name": "PROFILE_NAME",
        "Specie": "orig_name",
        "Cas": "CAS",
        "CAS#": "CAS",
        "Relative Mass": "weight_fraction",
        "Specie Id": "SPECIES_ID"
    }.items() if k in df.columns
})

# Make sure weight_fraction is numeric
df["weight_fraction"] = pd.to_numeric(df["weight_fraction"], errors="coerce")

missing_cols = {"PROFILE_CODE", "PROFILE_NAME", "orig_name", "CAS", "weight_fraction", "SPECIES_ID"} - set(df.columns)
assert not missing_cols, f"Expected columns not found: {missing_cols}"

print(f"Renamed SPECIEUROPE columns:")
for col in df.columns:
    print(col)
print()

Input SPECIEUROPE shape: 
13,148 rows × 22 columns

Input SPECIEUROPE columns:
Id
Name
Specie
Relative Mass
Uncertainty
Analythical Method
Uncertainty Method
Specie Id
Particle Size
Original Name
N Samples
Reference
Country
Place
Test Year
Profile Type
Sampling Method
Latitude
Longitude
Cas
Symbol
Unnamed: 21

Dropped empty columns: ['Unnamed: 21']

Renamed SPECIEUROPE columns:
PROFILE_CODE
PROFILE_NAME
orig_name
weight_fraction
Uncertainty
Analythical Method
Uncertainty Method
SPECIES_ID
Particle Size
Original Name
N Samples
Reference
Country
Place
Test Year
Profile Type
Sampling Method
Latitude
Longitude
CAS
Symbol



## Step 2: Clean names and CAS values and remove mixtures

This cell works on a copy (`df_rename`) and keeps the original in `orig_name_Original`. In order:

1. **Odd characters:** removes runs of `»` from `CAS` and `orig_name`, and strips whitespace.
2. **Synonyms:** in `orig_name`, cuts everything from ` (or` or ` ||` onwards, keeping only the first name.
3. **Mixtures by name:** drops rows whose `orig_name` contains `;`, `/` or `+`.
4. **Mixtures by CAS:** drops rows whose `CAS` contains the same separators.

Each stage prints counts of rows affected. The cleaned table is `df_clean_cas`.

In [18]:
# Check Name & CAS for odd values and edit the name
print("Checking 'orig_name' column for odd values.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")

# Copy df
df_rename = df.copy()

# Keep original names for diagnostics (placed right before the cleaned Name)
df_rename["orig_name_Original"] = df_rename["orig_name"]

cols = list(df_rename.columns)
cols.remove("orig_name_Original")
name_idx = cols.index("orig_name")
cols.insert(name_idx, "orig_name_Original")
df_rename = df_rename[cols]

# Total number of rows
n_total = len(df_rename)

# Remove '»', keep the first synonym, then drop mixtures (names and CAS with ; / +)
# (the CSV is read with its own dtypes, so the columns are converted to text first)
df_clean_cas = clean_names(
    df_rename,
    name_cols=["orig_name"],
    cas_cols=["CAS"],
    separators=[";", "/", "+"],
    force_text=True,
)


Checking 'orig_name' column for odd values. 2026-11-18 22:17:54 

Column 'CAS': '»' removed from 0 rows
Column 'orig_name': '»' removed from 0 rows

Total 'orig_name' values: 
13,148

'orig_name' containing synonym patterns (to be fixed): 
0

'orig_name' actually fixed: 
0

Total rows before Names filtration: 
13,148

Names containing mixture patterns (to be removed): 
0

Names actually removed: 
0

Remaining rows after filtering: 
13,148

Number of Names containing each separator pattern:
  ';': 0
  '/': 0
  '+': 0

Total rows before CAS# filtration: 
13,148

CAS# containing mixture patterns (to be removed): 
0

CAS# actually removed: 
0

Remaining rows after filtering: 
13,148

Number of CAS# containing each separator pattern:
  ';': 0
  '/': 0
  '+': 0

Filtered DF shape: 
13,148 rows × 22 columns



## Step 3: Format CAS numbers

SPECIEUROPE already provides hyphenated CAS numbers, so no reformatting is needed. This step copies `CAS` to the new `cas` column (placed right after `CAS`, as in the other databases), strips whitespace, and turns empty strings and the placeholders `none`/`nan` into real missing values. Values that do not look like a CAS number (`digits-2 digits-digit`) are reported but kept, because the PubChem lookup in step 4 falls back to the name.

Note: missing values stay `NaN` (a real null). The text conversion (`.astype(str)`, which turns `NaN` into `"nan"`) is applied later, when hashing for the PubChem cache and before the exports, so that Excel does not convert CAS numbers.

In [19]:
# Format CAS (missing values stay NaN)
print("Formatting CAS.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")

df_wf = df_clean_cas.copy()

df_wf["cas"] = normalize_cas(df_wf["CAS"])

cols_df_wf = list(df_wf.columns)
cas_idx = cols_df_wf.index("CAS")
cols_df_wf.remove("cas")
cols_df_wf.insert(cas_idx + 1, "cas")
df_wf = df_wf[cols_df_wf].copy()

n_odd = count_odd_cas(df_wf["cas"])
print(f"Rows without CAS#: {df_wf['cas'].isna().sum():,}")
print(f"Rows with a CAS# that does not look like a CAS number: {n_odd:,}\n")

print("Done formatting CAS#.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")
print(f"New CAS# column: \n{df_wf['cas']}\n")

Formatting CAS. 2026-11-18 22:17:54 

Rows without CAS#: 411
Rows with a CAS# that does not look like a CAS number: 7

Done formatting CAS#. 2026-11-18 22:17:54 

New CAS# column: 
0        7429-90-5
1        7440-36-0
2        7440-38-2
3        7440-43-9
4        7440-70-2
           ...    
13143    7440-32-6
13144    7440-62-2
13145    7440-66-6
13146    7440-33-7
13147    7440-42-8
Name: cas, Length: 13148, dtype: object



## Step 4: Retrieve compound properties from PubChem

This step replaces the external PubChem_Retriever / Data_Retriever tools and works as in the FooDB and SPECIATE notebooks: requests go to the PubChem PUG REST API, are rate-limited (`MAX_REQ_PER_SEC`, with extra delay if PubChem's `X-Throttling-Control` header reports load), and results are written to a cache file as they come in, so the step can be interrupted and re-run.

Each distinct combination of `orig_name`, `SPECIES_ID` and `cas` gets a hash-based `uid`. For each one the lookup tries, in order, until PubChem returns a hit:

1. `cas` (PubChem resolves CAS numbers as synonyms)
2. `orig_name`

The method that succeeded is stored in `Lookup_method`. Compounds PubChem cannot resolve are cached as `not_found`, so they are not retried on re-runs (delete their rows from the cache to retry). Compounds that fail because of a transient error are not cached and are retried on the next run.

Caveats: the first hit is taken when a query matches several PubChem compounds, and a CAS or name lookup can match a different compound than intended. Set `LOOKUP_LIMIT` in the configuration cell for a quick test run.

The first cell defines the helper functions; the second runs the lookup.

In [20]:
# PubChem client (rate limiter and request helpers live in inquire_common/pubchem.py)
client = PubChemClient(max_req_per_sec=MAX_REQ_PER_SEC)

def lookup_compound(row):
    """row: (uid, cas, name). Tries CAS# and Name in order."""
    uid, cas, name = row
    return lookup_attempts(uid, [
        ("CAS", cas, client.name_lookup_row),
        ("Name", name, client.name_lookup_row),
    ])

print(f"PubChem helpers defined | {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")


PubChem helpers defined | 2026-11-18 22:17:54



In [21]:
# Retrieve PubChem data for the unique identifier sets (resumable; results are appended to the cache)
print("Creating unique 'orig_name' & 'SPECIES_ID' & 'cas' set for PubChem data retrieval.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")

# Distinct identifier sets; the uid is a hash of the values, so the cache stays valid if row order changes
UNIQUE_KEY_COLS = ["orig_name", "SPECIES_ID", "cas"]

# Text form of CAS# (missing -> 'nan') in both df_wf and df_unique, so the uid hash is stable and the
# step 6 merge on the key columns matches.
df_wf["cas"] = df_wf["cas"].astype(str)

df_unique = add_uid(df_wf, UNIQUE_KEY_COLS)
print(f"Unique identifier sets: {len(df_unique):,}\n")

cache_cols = ["uid", "CID", "Name_PubChem", "IUPACName_PubChem", "InChIKey_PubChem", "Lookup_method"]
cache_current = run_cached_lookup(
    df_unique, ["cas", "orig_name"], per_row(lookup_compound, N_WORKERS),
    PUBCHEM_CACHE_CSV, cache_cols, client, chunk=CHUNK, limit=LOOKUP_LIMIT,
)


Creating unique 'orig_name' & 'SPECIES_ID' & 'cas' set for PubChem data retrieval. 2026-11-18 22:17:54 

Unique identifier sets: 232

232 of 232 compounds already cached, 0 to look up | 2026-11-18 22:17:54
No up-front time estimate: an Estimated Time of Arrival based on measured throughput is shown while running.
It is safe to interrupt and re-run.



Cached compounds: 232 of 232
Resolved to an InChIKey: 209
Lookup method:
Lookup_method
CAS          206
not_found     23
Name           3

PubChem retrieval done | 2026-11-18 22:17:55



## Step 5: Save the cleaned intermediate table (optional)

Writes the full cleaned table (`df_wf`) to CSV (`INTERMEDIATE_CSV` in the configuration cell), before InChIKey/keyword filtering.

In [22]:
# Write cleaned SPECIEUROPE to CSV file (path set in the configuration cell)
print("Writing filtered SPECIEUROPE to CSV.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")

# Prevent Excel CAS# conversion
df_wf["cas"] = df_wf["cas"].astype(str)
# Save SPECIEUROPE BEFORE further filtration
df_wf.to_csv(INTERMEDIATE_CSV, index=False)

print(f"Filtered SPECIEUROPE written to {INTERMEDIATE_CSV}.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")

print(f"Filtered DF shape: \n{df_wf.shape[0]:,} rows × {df_wf.shape[1]} columns\n")

Writing filtered SPECIEUROPE to CSV. 2026-11-18 22:17:55 

Filtered SPECIEUROPE written to ../../../data/cleaned_source_databases/specieurope/cleaned/SPECIEUROPE_Profiles_All_01_Fltrd.csv. 2026-11-18 22:17:55 

Filtered DF shape: 
13,148 rows × 23 columns



QC print of the columns in the intermediate table.

In [23]:
# QC print
print(f"Filtered SPECIEUROPE (intermediate) columns:")
for col in df_wf.columns:
    print(col)
print()

Filtered SPECIEUROPE (intermediate) columns:
PROFILE_CODE
PROFILE_NAME
orig_name_Original
orig_name
weight_fraction
Uncertainty
Analythical Method
Uncertainty Method
SPECIES_ID
Particle Size
Original Name
N Samples
Reference
Country
Place
Test Year
Profile Type
Sampling Method
Latitude
Longitude
CAS
cas
Symbol



## Step 6: Merge the PubChem data into the SPECIEUROPE table

Joins the cache from step 4 onto the full table (`df_wf`) using the three identifier columns, giving `df_xl`. Three PubChem columns are added, named so that the filtering below works unchanged:

| Column | Content |
| --- | --- |
| `name` | PubChem title of the matched compound |
| `inchikey` | PubChem InChIKey |
| `IUPACName` | PubChem IUPAC name |

`CID`, the PubChem compound ID, and `Lookup_method` (`CAS`, `Name` or `not_found`) are added too. `validate="m:1"` makes the merge fail if the lookup table were not unique per identifier set, and the row count is asserted to be unchanged. Compounds that PubChem could not resolve keep missing values and are removed in step 7a.

In [24]:
# Merge the PubChem results (cache from step 4) back onto the full table
print("Merging PubChem data into SPECIEUROPE.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")

pubchem = (
    read_cache(PUBCHEM_CACHE_CSV)
    .rename(columns={
        "Name_PubChem": "name",
        "IUPACName_PubChem": "IUPACName",
        "InChIKey_PubChem": "inchikey",
    })
)

# One row per unique identifier set, now with the PubChem columns
df_unique_pc = df_unique.merge(pubchem, on="uid", how="left", validate="1:1")
n_missing_lookup = df_unique_pc["Lookup_method"].isna().sum()
if n_missing_lookup:
    print(f"WARNING: {n_missing_lookup:,} unique compounds have no cached result yet. Re-run step 4 to retrieve them.\n")

# Many-to-one join back onto the full table (rows with missing key values match each other in pandas merges)
df_xl = df_wf.merge(
    df_unique_pc.drop(columns="uid"),
    on=UNIQUE_KEY_COLS,
    how="left",
    validate="m:1",
)
assert len(df_xl) == len(df_wf), "Row count changed during the PubChem merge"

print(f"Rows with InChIKey: {df_xl['inchikey'].notna().sum():,} of {len(df_xl):,}")
print(f"\nSPECIEUROPE + PubChem shape: \n{df_xl.shape[0]:,} rows × {df_xl.shape[1]} columns\n")
print("Done merging PubChem data.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")

Merging PubChem data into SPECIEUROPE. 2026-11-18 22:17:55 

Rows with InChIKey: 12,753 of 13,148

SPECIEUROPE + PubChem shape: 
13,148 rows × 28 columns

Done merging PubChem data. 2026-11-18 22:17:55 



QC print of the columns in the retrieved-data table. The filtering step below requires `inchikey` and `name` among them.

In [25]:
# QC print
print(f"Updated SPECIEUROPE columns:")
for col in df_xl.columns:
    print(col)
print()

Updated SPECIEUROPE columns:
PROFILE_CODE
PROFILE_NAME
orig_name_Original
orig_name
weight_fraction
Uncertainty
Analythical Method
Uncertainty Method
SPECIES_ID
Particle Size
Original Name
N Samples
Reference
Country
Place
Test Year
Profile Type
Sampling Method
Latitude
Longitude
CAS
cas
Symbol
CID
name
IUPACName
inchikey
Lookup_method



## Step 7a: Remove non-individual-chemical entries

Filters the table in three stages, each printing what was dropped:

1. **No InChIKey:** rows without an InChIKey are removed. These are mixtures or unidentified features that PubChem could not resolve to a single structure.
2. **Specific InChIKeys:** rows whose InChIKey is in the shared list `inquire_common/filters/drop_inchikeys.txt` are removed (elements, inorganic ions and salts, acids, isotopically labelled standards, polymers and similar). The trailing comments in the list give the compound each key stands for.
3. **Keywords in `name`:** rows whose name matches a keyword (metals, minerals, inorganic compounds, mixtures and biological materials) are removed. The keywords come from `keywords_common.txt` (shared by all sources) plus `keywords_specieurope.txt`, minus the protected names in `keyword_exceptions.txt`. Single-word terms get `\b` word boundaries so that, for example, `lead` does not match `leading`; multi-word terms are matched as-is. The match is case-insensitive.

The keyword and InChIKey lists are curated by hand, so review them against your own scope before relying on the output.

In [26]:
# Filter database by InChIKey & keywords
print("Filtering SPECIEUROPE by InChIKey & other keywords.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")

# Drops rows without InChIKey (mixtures or unidentified features), listed InChIKeys (mixtures, inorganic compounds,
# etc.) and unwanted name keywords. Lists live in inquire_common/filters: drop_inchikeys.txt,
# keywords_common.txt + keywords_specieurope.txt, keyword_exceptions.txt.
# anchor_end=True keeps whole-word matching; False also matches stems (magnes, mercur, ...) and removes more entries.
initial_n = len(df_xl)
df_inchikey, dropped_df, dropped_chems = apply_standard_filters(
    df_xl, source="specieurope", anchor_end=True, label="SPECIEUROPE",
)

print("Done keyword-filtering SPECIEUROPE.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")


Filtering SPECIEUROPE by InChIKey & other keywords. 2026-11-18 22:17:55 

Number of entries in SPECIEUROPE before InChIKey & keyword filtration: 
13,148

Dropped 395 entries with no InChIKey values. 

Remaining rows: 
12,753

Dropped compounds by InChIKey:
ATJFFYVFTNAWJD-UHFFFAOYSA-N >>> Tin atom
BVKZGUZCCUSVTD-UHFFFAOYSA-L >>> Carbonate Ion
KEAYESYHFKHZAL-UHFFFAOYSA-N >>> Sodium
NBIIXXVUZAFLBC-UHFFFAOYSA-K >>> Phosphate
NHNBFGGVMKEFGY-UHFFFAOYSA-N >>> Nitrate
NINIDFKCEFEMDL-UHFFFAOYSA-N >>> Sulfur
OAICVXFJPJFONN-UHFFFAOYSA-N >>> Phosphorus
OKTJSMMVPCPJKN-UHFFFAOYSA-N >>> Carbon
QAOWNCQODCNURD-UHFFFAOYSA-L >>> Sulfate Ion
QGZKDVFQNNGYKY-UHFFFAOYSA-O >>> Ammonium
QTBSBXVTEAMEQO-UHFFFAOYSA-M >>> Acetate
VEXZGXHMUGYJMC-UHFFFAOYSA-M >>> Chloride
ZLMJMSJWJFRBEC-UHFFFAOYSA-N >>> Potassium

Dropped 13 unique InChIKeys (2,245 rows)
Rows remaining: 10,508

Dropped unique names by keyword: 42

Alumina
Aluminum
Antimony
Arsenic
Atomic chlorine
Barium
Bismuth
Boron
Cadmium
Calcium
Calcium Cation
C

### Step 7b: Sort and save the filtered SPECIEUROPE

Sorts the filtered table by `PROFILE_CODE`, then `weight_fraction`, then `name` (a single multi-key `sort_values` call; later keys only break ties in earlier ones). It then writes the table to CSV (`FLTRD_CSV` in the configuration cell).

Earlier versions called `sort_values` three times in a row, one key each, which meant only the last call (`PROFILE_CODE`) determined the order. Sorting is ascending, and rows with missing values in a key are placed last.

In [27]:
# Write keyword-filtered SPECIEUROPE to CSV file (path set in the configuration cell)
print("Sorting keyword-filtered SPECIEUROPE.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")

# Sort by all three keys at once (first key is the primary key, later keys break ties)
df_inchikey = df_inchikey.sort_values(
    ["PROFILE_CODE", "weight_fraction", "name"],
    kind="stable",
)

print("Writing keyword-filtered SPECIEUROPE to CSV.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")

# Prevent Excel CAS# conversion
df_inchikey["cas"] = df_inchikey["cas"].astype(str)
# Save SPECIEUROPE AFTER the InChIKey & name/keyword filtration
df_inchikey.to_csv(FLTRD_CSV, index=False, encoding="utf-8")

print(f"Keyword-filtered SPECIEUROPE written to {FLTRD_CSV}.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")
print(f"Keyword-filtered SPECIEUROPE CSV shape: \n{df_inchikey.shape[0]:,} rows × {df_inchikey.shape[1]} columns\n")

Sorting keyword-filtered SPECIEUROPE. 2026-11-18 22:17:55 

Writing keyword-filtered SPECIEUROPE to CSV. 2026-11-18 22:17:55 

Keyword-filtered SPECIEUROPE written to ../../../data/cleaned_source_databases/specieurope/cleaned/SPECIEUROPE_Profiles_All_02_Fltrd.csv. 2026-11-18 22:17:55 

Keyword-filtered SPECIEUROPE CSV shape: 
4,299 rows × 28 columns



QC print of the shape and columns of the final filtered table.

In [28]:
# QC print
print(f"Keyword-filtered SPECIEUROPE shape: \n{df_inchikey.shape[0]:,} rows × {df_inchikey.shape[1]} columns\n")

print(f"Keyword-filtered SPECIEUROPE columns:")
for col in df_inchikey.columns:
    print(col)
print()

Keyword-filtered SPECIEUROPE shape: 
4,299 rows × 28 columns

Keyword-filtered SPECIEUROPE columns:
PROFILE_CODE
PROFILE_NAME
orig_name_Original
orig_name
weight_fraction
Uncertainty
Analythical Method
Uncertainty Method
SPECIES_ID
Particle Size
Original Name
N Samples
Reference
Country
Place
Test Year
Profile Type
Sampling Method
Latitude
Longitude
CAS
cas
Symbol
CID
name
IUPACName
inchikey
Lookup_method



### Step 8: Descriptive statistics of the filtered SPECIEUROPE

The next cells compute summary statistics on the final filtered table (`df_inchikey`). The first cell shows per-column dtype, unique and missing counts (`df_stats` from `inquire_common`) and creates the empty accumulators (sets, dicts, counters) that the following cells fill.

In [29]:
# Statistics accumulators
print("Creating statistics accumulators.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")

df_stat = df_inchikey

_ = df_stats(df_stat, "Filtered SPECIEUROPE")

total_rows = len(df_stat)

unique_profiles = set()
unique_orig_names = set()
unique_feat_names = set()

casrn_profile_stats = defaultdict(lambda: {
    "name": None,
    "row_count": 0,
    "profiles": set(),
    "weight_fraction": []
})

column_null_counts = {}
overall_values_per_column = {}
unique_values_per_column = {}

value_counts_limited = {}  # optional sample column

print("Statistics accumulators created.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")

Creating statistics accumulators. 2026-11-18 22:17:55 


Filtered SPECIEUROPE
Rows: 4,299
Columns: 28


,column,dtype,n_unique,n_missing,pct_missing
4,weight_fraction,float64,2355,271,6.30
5,Uncertainty,float64,2048,681,15.84
0,PROFILE_CODE,int64,422,0,0.00
10,Original Name,object,292,0,0.00
1,PROFILE_NAME,object,251,0,0.00
2,orig_name_Original,object,154,0,0.00
3,orig_name,object,154,0,0.00
8,SPECIES_ID,int64,154,0,0.00
21,cas,object,154,0,0.00
23,CID,Int64,153,0,0.00


Statistics accumulators created. 2026-11-18 22:17:56 



Column-level statistics: for every column, the set of distinct non-null values, the number of non-null values and the number of nulls.

In [30]:
# Column-level statistics
print("Calculating column-level statistics.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")

for col in df_stat.columns:
    non_null = df_stat[col].dropna()

    unique_values_per_column[col] = set(non_null.unique())
    overall_values_per_column[col] = non_null.shape[0]
    column_null_counts[col] = df_stat[col].isna().sum()

print("Column-level statistics calculated.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")

Calculating column-level statistics. 2026-11-18 22:17:56 

Column-level statistics calculated. 2026-11-18 22:17:56 



Collect the distinct profile names, original compound names (`orig_name`) and PubChem-based names (`name`) in the filtered table.

In [31]:
# Global unique entities
print("Calculating global unique entities.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")

unique_profiles.update(df_stat["PROFILE_NAME"].dropna().unique())
unique_orig_names.update(df_stat["orig_name"].dropna().unique())
unique_feat_names.update(df_stat["name"].dropna().unique())

print("Global unique entities calculated.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")

Calculating global unique entities. 2026-11-18 22:17:56 

Global unique entities calculated. 2026-11-18 22:17:56 



Aggregate per `cas`: row count, first-seen feature name, set of profiles it occurs in, and the list of all `weight_fraction` values. Rows without a `cas` (NaN, or the text `nan`/`None`) are skipped and counted. This uses `iterrows()`, so it can be slow on large tables.

In [32]:
# CAS# → profile/weight fraction relationships
print("Establishing CAS# & profile/weight fraction relationships.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")

n_skipped_no_cas = 0

for _, row in df_stat.iterrows():
    casrn = row.get("cas")

    # Missing CAS# is NaN, or the text 'nan' after the astype(str) before the exports
    if pd.isna(casrn) or str(casrn).strip().lower() in ("nan", "none", ""):
        n_skipped_no_cas += 1
        continue

    s = casrn_profile_stats[casrn]

    # Row-level count
    s["row_count"] += 1

    # Chemical name (store once)
    chem_name = row.get("name")
    if pd.notna(chem_name) and s["name"] is None:
        s["name"] = chem_name

    # Profile names
    profile = row.get("PROFILE_NAME")
    if pd.notna(profile):
        s["profiles"].add(profile)

    # Weight fractions
    cwf = row.get("weight_fraction")
    if pd.notna(cwf):
        s["weight_fraction"].append(cwf)

print(f"Skipped {n_skipped_no_cas:,} rows without a CAS#.")
print(f"{len(casrn_profile_stats):,} distinct CAS# aggregated.\n")
print("CAS# & profile/weight fraction relationships established.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")

Establishing CAS# & profile/weight fraction relationships. 2026-11-18 22:17:56 

Skipped 27 rows without a CAS#.
153 distinct CAS# aggregated.

CAS# & profile/weight fraction relationships established. 2026-11-18 22:17:56 



Summary table: for every column, the number of non-null values ("Overall"), the number of distinct values ("Unique") and the total number of rows.

In [33]:
# Summary
print("===== OVERALL & UNIQUE VALUE COUNTS PER COLUMN =====\n")

rows = []

for col in unique_values_per_column:
    rows.append({
        "Column": col,
        "Overall": overall_values_per_column.get(col, 0),
        "Unique": len(unique_values_per_column[col]),
        "Total rows": total_rows
    })

summary_df = pd.DataFrame(rows)

summary_df.style.format({
    "Overall": "{:,}",
    "Unique": "{:,}",
    "Total rows": "{:,}"
}).set_table_styles([
    {"selector": "th, td", "props": [("border", "1px solid black"), ("text-align", "left")]},
    {"selector": "table", "props": [("border-collapse", "collapse")]}
])

===== OVERALL & UNIQUE VALUE COUNTS PER COLUMN =====



,Column,Overall,Unique,Total rows
0,PROFILE_CODE,"4,299",422,"4,299"
1,PROFILE_NAME,"4,299",251,"4,299"
2,orig_name_Original,"4,299",154,"4,299"
3,orig_name,"4,299",154,"4,299"
4,weight_fraction,"4,028","2,355","4,299"
5,Uncertainty,"3,618","2,048","4,299"
6,Analythical Method,"3,967",15,"4,299"
7,Uncertainty Method,"3,978",5,"4,299"
8,SPECIES_ID,"4,299",154,"4,299"
9,Particle Size,"4,299",7,"4,299"


Show the `TOP_N` most frequently occurring compounds (by number of rows), with their number of distinct profiles and weight-fraction min/max/mean.

In [34]:
# Show top N most frequent compounds
TOP_N = 30

# Sort CAS# by row_count descending
sorted_casrn = sorted(
    casrn_profile_stats.items(),
    key=lambda x: x[1]["row_count"],
    reverse=True
)

# Build table rows
rows = []

for casrn, stats in sorted_casrn[:TOP_N]:
    chem_name = stats["name"] or "Unknown"
    row_count = stats["row_count"]
    n_profiles = len(stats["profiles"])

    # Weight fraction stats
    if stats["weight_fraction"]:
        wf_min = np.min(stats["weight_fraction"])
        wf_max = np.max(stats["weight_fraction"])
        wf_mean = np.mean(stats["weight_fraction"])
    else:
        wf_min = wf_max = wf_mean = np.nan

    rows.append({
        "cas": casrn,
        "Chemical name": chem_name,
        "Occurrences (rows)": row_count,
        "Unique profiles": n_profiles,
        "WF min": wf_min,
        "WF max": wf_max,
        "WF mean": wf_mean
    })

# Create DataFrame
casrn_df = pd.DataFrame(rows)

# Render nicely with commas and borders
casrn_df.style.format({
    "Occurrences (rows)": "{:,}",
    "Unique profiles": "{:,}",
    "WF min": "{:.5g}",
    "WF max": "{:.5g}",
    "WF mean": "{:.5g}"
}).set_table_styles([
    {"selector": "th, td", "props": [("border", "1px solid black"), ("text-align", "left")]},
    {"selector": "table", "props": [("border-collapse", "collapse")]}
])

,cas,Chemical name,Occurrences (rows),Unique profiles,WF min,WF max,WF mean
0,7439-96-5,Manganese,401,234,-0.00038,0.39774,0.0024539
1,7440-62-2,Vanadium,377,238,-0.00014,0.094992,0.0011799
2,7440-32-6,Titanium,299,185,-0.0005,0.028161,0.001136
3,17341-25-2,Sodium Ion,205,86,0,0.39339,0.040018
4,7439-95-4,Magnesium,200,163,0,0.53,0.013855
5,24203-36-9,Potassium Cation,192,73,0,0.111,0.0069001
6,22537-22-0,Magnesium Cation,192,73,0,0.25528,0.0059922
7,7439-98-7,Molybdenum,181,85,-1.3e-05,0.00957,0.00018942
8,498-07-7,Levoglucosan,167,45,0,0.151,0.014931
9,14168-45-1,"1,6-anhydro-beta-D-mannopyranose",153,36,0,0.04,0.002029


### Step 9: Export per-compound statistics

Writes one row per `cas` (feature name, number of rows, number of distinct profiles, and min/mean/max `weight_fraction`) from `casrn_profile_stats` to CSV. The output path (`STATS_CSV`) is set in the configuration cell.

In [35]:
# Write stats DF to CSV file (path set in the configuration cell)
print("Writing stats DF to CSV.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")

rows = []
for casrn, s in casrn_profile_stats.items():
    rows.append({
        "cas": casrn,
        "name": s["name"] or "Unknown",
        "Row count": s["row_count"],
        "Unique profiles": len(s["profiles"]),
        "WF mean": np.mean(s["weight_fraction"]) if s["weight_fraction"] else np.nan,
        "WF min": min(s["weight_fraction"]) if s["weight_fraction"] else np.nan,
        "WF max": max(s["weight_fraction"]) if s["weight_fraction"] else np.nan,
    })

pd.DataFrame(rows).to_csv(STATS_CSV, index=False, encoding="utf-8")

print(f"Stats DF written to {STATS_CSV}.", datetime.now().strftime('%Y-%m-%d %H:%M:%S'), "\n")

Writing stats DF to CSV. 2026-11-18 22:17:57 

Stats DF written to ../../../data/cleaned_source_databases/specieurope/cleaned/SPECIEUROPE_Profiles_All_03_CAS_Profile_Composition.csv. 2026-11-18 22:17:57 

